# Bronze JSON Explorer

Notebook para leer los JSON de Bronze con una narrativa ejecutiva. Primero inventaria los archivos, luego normaliza las estructuras y por ultimo deja una base limpia para EDA y preparacion de Gold.

La lectura busca responder dos preguntas: que campos aportan valor analitico real y que partes deben quedarse como trazabilidad o contexto tecnico.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'datalake_bronze').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
ENV_FILE = PROJECT_ROOT / 'airflow' / '.env'
DEFAULT_BRONZE_DIR = PROJECT_ROOT / 'datalake_bronze'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'ENV_FILE = {ENV_FILE}')
print(f'DEFAULT_BRONZE_DIR = {DEFAULT_BRONZE_DIR}')

PROJECT_ROOT = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming
ENV_FILE = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/airflow/.env
DEFAULT_BRONZE_DIR = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/datalake_bronze


In [2]:
def load_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def resolve_bronze_dir(raw_value: str | Path | None) -> Path:
    if raw_value is None:
        return DEFAULT_BRONZE_DIR
    candidate = Path(raw_value)
    if not candidate.is_absolute():
        candidate = (PROJECT_ROOT / candidate).resolve()
    if candidate.exists():
        return candidate
    return DEFAULT_BRONZE_DIR


env = load_env_file(ENV_FILE)
bronze_dir = resolve_bronze_dir(env.get('BRONZE_BASE_PATH'))
bronze_files = sorted(bronze_dir.glob('*.json'))
print(f'DIRECTORIO BRONZE = {bronze_dir}')
print(f'Archivos JSON encontrados = {len(bronze_files)}')
display(pd.DataFrame({'bronze_json': [path.name for path in bronze_files]}))

DIRECTORIO BRONZE = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/datalake_bronze
Archivos JSON encontrados = 11


,bronze_json
0,twitter_tweets_20260529_034202.json
1,twitter_tweets_20260529_231325.json
2,twitter_tweets_20260601_204302.json
3,twitter_tweets_20260601_204412.json
4,twitter_tweets_20260602_223250.json
5,webscraping_noticias_20260529_040828.json
6,webscraping_noticias_20260529_231325.json
7,webscraping_noticias_20260601_204302.json
8,webscraping_noticias_20260601_204415.json
9,webscraping_noticias_20260602_014737.json


## 1. Normalizacion de Bronze

En este bloque se convierten los JSON de Bronze en tablas de pandas mas faciles de leer y auditar. El objetivo no es solo abrir el archivo, sino separar rapidamente snapshots, noticias, comentarios y tweets.

In [3]:
def load_json_payload(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def infer_source(path: Path) -> str:
    name = path.name.lower()
    if 'webscraping' in name:
        return 'webscraping'
    if 'twitter' in name:
        return 'twitter'
    return 'unknown'


def ensure_list(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        return [payload]
    return []


def build_webscraping_tables(path: Path) -> dict[str, pd.DataFrame]:
    payload = ensure_list(load_json_payload(path))
    snapshots = pd.json_normalize(payload, sep='_')
    snapshots['bronze_file'] = path.name
    if 'date' in snapshots.columns:
        snapshots['date_parsed'] = pd.to_datetime(pd.to_numeric(snapshots['date'], errors='coerce'), unit='ms', errors='coerce')
    if 'createdAt' in snapshots.columns:
        snapshots['createdAt'] = pd.to_datetime(snapshots['createdAt'], errors='coerce')
    if 'updatedAt' in snapshots.columns:
        snapshots['updatedAt'] = pd.to_datetime(snapshots['updatedAt'], errors='coerce')
    if 'news' not in snapshots.columns:
        return {'webscraping_snapshots': snapshots}

    snapshots['news_count'] = snapshots['news'].apply(lambda value: len(value) if isinstance(value, list) else 0)
    news_rows = snapshots.explode('news', ignore_index=True)
    news_details = pd.json_normalize(news_rows['news']).add_prefix('news_')
    news_table = pd.concat([news_rows.drop(columns=['news']), news_details], axis=1)
    if 'news_comments' in news_table.columns:
        news_table['news_comments_count'] = news_table['news_comments'].apply(lambda value: len(value) if isinstance(value, list) else 0)
        comments_table = news_table.explode('news_comments', ignore_index=True).rename(columns={'news_comments': 'comment_text'})
        comments_table['comment_text'] = comments_table['comment_text'].astype('string')
    else:
        comments_table = news_table.copy()

    return {
        'webscraping_snapshots': snapshots.drop(columns=['news']),
        'webscraping_news': news_table,
        'webscraping_comments': comments_table,
    }


def build_twitter_tables(path: Path) -> dict[str, pd.DataFrame]:
    payload = ensure_list(load_json_payload(path))
    tweets = pd.json_normalize(payload, sep='_')
    tweets['bronze_file'] = path.name
    if 'createdAt' in tweets.columns:
        tweets['createdAt'] = pd.to_datetime(tweets['createdAt'], errors='coerce')
    if 'text' in tweets.columns:
        tweets['text_length'] = tweets['text'].fillna('').astype(str).str.len()
    return {'twitter_tweets': tweets}

In [4]:
bronze_tables: dict[str, list[pd.DataFrame]] = {}

for bronze_file in bronze_files:
    source = infer_source(bronze_file)
    if source == 'webscraping':
        tables = build_webscraping_tables(bronze_file)
    elif source == 'twitter':
        tables = build_twitter_tables(bronze_file)
    else:
        continue

    for table_name, table in tables.items():
        bronze_tables.setdefault(table_name, []).append(table)

bronze_tables = {table_name: pd.concat(parts, ignore_index=True) for table_name, parts in bronze_tables.items()}

for table_name, table in bronze_tables.items():
    print(f'{table_name}: {table.shape[0]} filas x {table.shape[1]} columnas')

twitter_tweets: 5 filas x 7 columnas
webscraping_snapshots: 6 filas x 8 columnas
webscraping_news: 40 filas x 12 columnas
webscraping_comments: 838 filas x 12 columnas


## 2. Vista rapida y primeras lecturas

En esta seccion se revisan columnas, tipos, valores faltantes y unas primeras filas por cada tabla para identificar rapidamente que tan util es cada bloque de Bronze.

In [5]:
for table_name, table in bronze_tables.items():
    display(Markdown(f'### {table_name}'))
    display(table.head(5))
    display(table.dtypes.astype(str).to_frame('dtype'))
    missing = table.isna().mean().sort_values(ascending=False).to_frame('missing_ratio')
    display(missing.head(15))

### twitter_tweets

,_id,tweets,date,createdAt,updatedAt,__v,bronze_file
0,6a1851291ca1ed31ad6cd431,"[{'type': 'tweet', 'id': '2060004476669374743'...",1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260529_034202.json
1,6a1851291ca1ed31ad6cd431,"[{'type': 'tweet', 'id': '2060004476669374743'...",1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260529_231325.json
2,6a1851291ca1ed31ad6cd431,"[{'type': 'tweet', 'id': '2060004476669374743'...",1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260601_204302.json
3,6a1851291ca1ed31ad6cd431,"[{'type': 'tweet', 'id': '2060004476669374743'...",1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260601_204412.json
4,6a1f1e6ff02a8db79c3ec110,"[{'type': 'tweet', 'id': '2061811564177666186'...",1780424303180,2026-06-02 18:18:23.190,2026-06-02T18:18:23.190000,0,twitter_tweets_20260602_223250.json


,dtype
_id,str
tweets,object
date,str
createdAt,datetime64[us]
updatedAt,str
__v,int64
bronze_file,str


,missing_ratio
_id,0.0
tweets,0.0
date,0.0
createdAt,0.0
updatedAt,0.0
__v,0.0
bronze_file,0.0


### webscraping_snapshots

...

,missing_ratio
_id,0.0
date,0.0
createdAt,0.0
updatedAt,0.0
__v,0.0
bronze_file,0.0
date_parsed,0.0
news_count,0.0
news_newsLink,0.0
news_comments,0.0


### webscraping_comments

,_id,date,createdAt,updatedAt,__v,bronze_file,date_parsed,news_count,news_newsLink,comment_text,news__id,news_comments_count
0,6a18a46df83441dbb99038b1,1779999853273,2026-05-28 20:24:13.283,2026-05-28 20:24:13.283,0,webscraping_noticias_20260529_040828.json,2026-05-28 20:24:13.273,6,https://www.gsmarena.com/us_smartphone_market_...,"Anonymous, 22 hours agoUS smartphone market ca...",6a18a46df83441dbb99038b2,16
1,6a18a46df83441dbb99038b1,1779999853273,2026-05-28 20:24:13.283,2026-05-28 20:24:13.283,0,webscraping_noticias_20260529_040828.json,2026-05-28 20:24:13.273,6,https://www.gsmarena.com/us_smartphone_market_...,"3% doesn't seem like much, to me. Sure, it's u...",6a18a46df83441dbb99038b2,16
2,6a18a46df83441dbb99038b1,1779999853273,2026-05-28 20:24:13.283,2026-05-28 20:24:13.283,0,webscraping_noticias_20260529_040828.json,2026-05-28 20:24:13.273,6,https://www.gsmarena.com/us_smartphone_market_...,"PatrickCzech, 15 hours agoiOS 26 is not buggy ...",6a18a46df83441dbb99038b2,16
3,6a18a46df83441dbb99038b1,1779999853273,2026-05-28 20:24:13.283,2026-05-28 20:24:13.283,0,webscraping_noticias_20260529_040828.json,2026-05-28 20:24:13.273,6,https://www.gsmarena.com/us_smartphone_market_...,"Anonymous, 15 hours agoYour boring ios is mile...",6a18a46df83441dbb99038b2,16
4,6a18a46df83441dbb99038b1,1779999853273,2026-05-28 20:24:13.283,2026-05-28 20:24:13.283,0,webscraping_noticias_20260529_040828.json,2026-05-28 20:24:13.273,6,https://www.gsmarena.com/us_smartphone_market_...,"PatrickCzech, 15 hours agoiOS 26 is not buggy ...",6a18a46df83441dbb99038b2,16


,dtype
_id,str
date,str
createdAt,datetime64[us]
updatedAt,datetime64[us]
__v,int64
bronze_file,str
date_parsed,datetime64[ms]
news_count,int64
news_newsLink,str
comment_text,string


,missing_ratio
_id,0.0
date,0.0
createdAt,0.0
updatedAt,0.0
__v,0.0
bronze_file,0.0
date_parsed,0.0
news_count,0.0
news_newsLink,0.0
comment_text,0.0


## 3. Analisis inicial

Estas celdas dejan listas algunas agregaciones basicas para empezar el analisis con pandas y entender donde esta la densidad real del dato.

In [6]:
if 'webscraping_comments' in bronze_tables:
    comments = bronze_tables['webscraping_comments'].copy()
    if 'date_parsed' in comments.columns:
        comments['day'] = comments['date_parsed'].dt.date
        display(comments.groupby('day').size().sort_values(ascending=False).head(10).to_frame('comments'))
    if 'news_newsLink' in comments.columns:
        display(comments['news_newsLink'].value_counts().head(10).to_frame('comments'))

if 'twitter_tweets' in bronze_tables:
    tweets = bronze_tables['twitter_tweets'].copy()
    if 'createdAt' in tweets.columns:
        tweets['day'] = tweets['createdAt'].dt.date
        display(tweets.groupby('day').size().sort_values(ascending=False).head(10).to_frame('tweets'))
    if 'author_name' in tweets.columns:
        display(tweets['author_name'].value_counts().head(10).to_frame('tweets'))

,comments
day,
2026-05-31,334
2026-06-02,170
2026-05-29,119
2026-05-28,109
2026-06-01,106


,comments
news_newsLink,
https://www.arenaev.com/xiaomi_ceo_personally_delivers_first_yu7_gt_vehicles_to_buyers_in_china-news-5928.php,195
https://www.arenaev.com/rivian_rejects_apple_and_bets_heavy_on_ai_for_future_electric_cars-news-5932.php,151
https://www.gsmarena.com/vertu_unveils_alphafold_its_first_bookstyle_foldable_and_boasts_about_its_hermes_ai_agent-news-73049.php,85
https://www.gsmarena.com/oppo_find_x10_honor_magic9_series_said_to_be_testing_a_100mp_square_selfie_sensor-news-73053.php,56
https://www.gsmarena.com/weekly_poll_results_android_17_with_better_security_and_airdrop_support_has_people_excited-news-73031.php,45
https://www.gsmarena.com/facebook_instagram_and_whatsapp_now_offer_subscription_plans-news-73020.php,31
https://www.gsmarena.com/xiaomi_sound_play_handson-news-73043.php,29
https://www.gsmarena.com/weekly_poll_will_you_buy_the_xiaomi_17t_or_the_xiaomi_17t_pro-news-73029.php,27
https://www.gsmarena.com/oppo_coloros_16_may_2026_update-news-73057.php,24


,tweets
day,
2026-05-28,4
2026-06-02,1


### Lectura ejecutiva del analisis inicial

- **Hallazgo principal:** las series por dia y las listas por autor o enlace ayudan a ubicar donde se concentra el volumen real del dato.
- **Dato relevante:** los picos suelen reflejar actividad puntual, no necesariamente calidad o valor de negocio.
- **Decision:** antes de modelar conviene revisar si el volumen viene de concentracion natural o de repeticion en la captura.

### Lectura ejecutiva del cierre

Bronze ya queda listo como base de exploracion y como insumo directo para construir Gold. El siguiente paso es conservar solo las columnas estables, documentar bien el grano y evitar joins que mezclen entidades sin necesidad.

## 4. Preparacion de tweets para EDA y NLP

En esta seccion se convierten los tweets Bronze en una tabla plana, se limpia el texto, se detecta idioma y se deja un CSV base completo para analisis de sentimiento y EDA.

In [7]:
import re
from pathlib import Path

try:
    from langdetect import DetectorFactory, LangDetectException, detect
except Exception:
    # Fallback simple detector when `langdetect` is not installed.
    class LangDetectException(Exception):
        pass

    class _DummyDetectorFactory:
        seed = None

    DetectorFactory = _DummyDetectorFactory()

    def detect(text: str) -> str:
        if not text or not isinstance(text, str):
            return 'unknown'
        s = ' ' + text.lower() + ' '
        eng = sum(s.count(w) for w in (' the ', ' and ', ' is ', ' to ', ' of ', ' in ', ' for ', ' you '))
        esp = sum(s.count(w) for w in (' que ', ' la ', ' el ', ' los ', ' las ', ' y ', ' de ', ' para '))
        if eng == 0 and esp == 0:
            if any(c in s for c in 'áéíóúñ'):
                return 'es'
            return 'unknown'
        return 'en' if eng >= esp else 'es'

DetectorFactory.seed = 42


TWEETS_OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'output'
TWEETS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TWEETS_CSV_PATH = TWEETS_OUTPUT_DIR / 'bronze_twitter_eda.csv'


snapshot_tweets = bronze_tables['twitter_tweets'].copy()
print(f'Filas de snapshot: {len(snapshot_tweets)}')
print(snapshot_tweets.columns.to_list())


def build_twitter_flat_table(snapshot_df: pd.DataFrame) -> pd.DataFrame:
    exploded = snapshot_df.explode('tweets', ignore_index=True)
    snapshot_part = exploded.drop(columns=['tweets']).add_prefix('snapshot_')
    tweet_details = pd.json_normalize(exploded['tweets'], sep='_').add_prefix('tweet_')
    flat = pd.concat([snapshot_part, tweet_details], axis=1)
    return flat


def clean_text(text: str | None) -> str:
    if text is None:
        return ''
    cleaned = str(text)
    cleaned = re.sub(r'https?://\S+|www\.\S+', ' ', cleaned)
    cleaned = re.sub(r'@\w+', ' ', cleaned)
    cleaned = re.sub(r'#', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def detect_language(text: str) -> str:
    if not text:
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'


tweets_flat = build_twitter_flat_table(snapshot_tweets)
print(f'Filas planas: {len(tweets_flat)}')
tweets_flat.head(3)

Filas de snapshot: 5
['_id', 'tweets', 'date', 'createdAt', 'updatedAt', '__v', 'bronze_file']
Filas planas: 300


,snapshot__id,snapshot_date,snapshot_createdAt,snapshot_updatedAt,snapshot___v,snapshot_bronze_file,tweet_type,tweet_id,tweet_url,tweet_twitterUrl,...,tweet_quoted_tweet_quoted_tweet_card,tweet_quoted_tweet_quoted_tweet_quoted_tweet,tweet_quoted_tweet_quoted_tweet_retweeted_tweet,tweet_quoted_tweet_quoted_tweet_isLimitedReply,tweet_quoted_tweet_quoted_tweet_communityInfo,tweet_quoted_tweet_quoted_tweet_article,tweet_author_profile_bio_entities_description_symbols,tweet_entities_timestamps,tweet_quoted_tweet_author_profile_bio_entities_description_hashtags,tweet_quoted_tweet_author_profile_bio_entities_url_urls
0,6a1851291ca1ed31ad6cd431,1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260529_034202.json,tweet,2060004476669374743,https://x.com/SouthernFreeez/status/2060004476...,https://twitter.com/SouthernFreeez/status/2060...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6a1851291ca1ed31ad6cd431,1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260529_034202.json,tweet,2059950505447584179,https://x.com/oluwafemi10591/status/2059950505...,https://twitter.com/oluwafemi10591/status/2059...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6a1851291ca1ed31ad6cd431,1779978537776,2026-05-28 14:28:57.780,2026-05-28T14:28:57.780000,0,twitter_tweets_20260529_034202.json,tweet,2059950392176234910,https://x.com/oluwafemi10591/status/2059950392...,https://twitter.com/oluwafemi10591/status/2059...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Asegura que tweets_selected exista aunque no se haya ejecutado la celda anterior
if 'tweets_selected' not in globals():
    cols = [c for c in ('tweet_text', 'tweet_lang') if c in tweets_flat.columns]
    tweets_selected = tweets_flat[cols].copy()
    if 'tweet_text' not in tweets_selected.columns:
        tweets_selected['tweet_text'] = ''
    if 'tweet_lang' not in tweets_selected.columns:
        tweets_selected['tweet_lang'] = None

# Asegura que existan las columnas derivadas necesarias
tweets_selected['tweet_text_raw'] = tweets_selected['tweet_text'].astype('string')
tweets_selected['clean_text'] = tweets_selected['tweet_text_raw'].map(clean_text)
tweets_selected['text_length'] = tweets_selected['clean_text'].str.len()
tweets_selected['detected_lang'] = tweets_selected['clean_text'].fillna('').map(detect_language)
tweets_selected['is_english'] = tweets_selected['tweet_lang'].eq('en') | tweets_selected['detected_lang'].eq('en')
tweets_selected['lang_match'] = tweets_selected['tweet_lang'].fillna('unknown').eq(tweets_selected['detected_lang'].fillna('unknown'))

lang_summary = (
    tweets_selected[['tweet_lang', 'detected_lang', 'is_english']]
    .assign(tweet_lang=lambda frame: frame['tweet_lang'].fillna('unknown'))
    .groupby(['tweet_lang', 'detected_lang', 'is_english'])
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
)

display(lang_summary)

print(f'Filas antes de exportar: {len(tweets_selected)}')
print(f'Filas con bandera de ingles: {int(tweets_selected["is_english"].sum())}')
print(f'Filas sin bandera de ingles: {int((~tweets_selected["is_english"]).sum())}')

tweets_eda = tweets_selected.copy()
tweets_eda.to_csv(TWEETS_CSV_PATH, index=False, encoding='utf-8')
print(f'CSV guardado en {TWEETS_CSV_PATH}')
tweets_eda.head(5)

,tweet_lang,detected_lang,is_english,rows
0,en,en,True,273
1,en,unknown,True,13
2,in,unknown,False,9
3,tl,unknown,False,4
4,und,en,True,1


Filas antes de exportar: 300
Filas con bandera de ingles: 287
Filas sin bandera de ingles: 13
CSV guardado en /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/notebooks/output/bronze_twitter_eda.csv


,tweet_text,tweet_lang,tweet_text_raw,clean_text,text_length,detected_lang,is_english,lang_match
0,@Bose 1/ Really disappointed with Bose ending ...,en,@Bose 1/ Really disappointed with Bose ending ...,1/ Really disappointed with Bose ending cloud ...,266,en,True,True
1,I got into a screaming match on the bus this m...,en,I got into a screaming match on the bus this m...,I got into a screaming match on the bus this m...,1064,en,True,True
2,I got into a screaming match on the bus this m...,en,I got into a screaming match on the bus this m...,I got into a screaming match on the bus this m...,1064,en,True,True
3,Ever forget you’re wearing earbuds until the b...,en,Ever forget you’re wearing earbuds until the b...,Ever forget you’re wearing earbuds until the b...,142,en,True,True
4,Dashy talks about Mercules' camera falling on ...,en,Dashy talks about Mercules' camera falling on ...,Dashy talks about Mercules' camera falling on ...,275,en,True,True


In [9]:
tweet_column_audit = pd.DataFrame({
    'column': tweets_flat.columns,
    'dtype': tweets_flat.dtypes.astype(str).values,
    'missing_ratio': tweets_flat.isna().mean().values,
}).sort_values(['missing_ratio', 'column'], ascending=[False, True])

display(tweet_column_audit.head(40))

analysis_columns = [
    'snapshot_bronze_file',
    'snapshot__id',
    'snapshot_has_next_page',
    'snapshot_next_cursor',
    'tweet_id',
    'tweet_text',
    'tweet_createdAt',
    'tweet_lang',
    'tweet_source',
    'tweet_retweetCount',
    'tweet_replyCount',
    'tweet_likeCount',
    'tweet_quoteCount',
    'tweet_viewCount',
    'tweet_isReply',
    'tweet_conversationId',
    'tweet_author_userName',
    'tweet_author_name',
    'tweet_author_id',
    'tweet_author_followers',
    'tweet_author_following',
    'tweet_author_isVerified',
    'tweet_author_isBlueVerified',
    'tweet_author_location',
    'tweet_author_createdAt',
]

analysis_columns = [column for column in analysis_columns if column in tweets_flat.columns]
columns_drop = [column for column in tweets_flat.columns if column not in analysis_columns]

print('Columnas que se conservan:', analysis_columns)
print('Columnas que se descartan:', columns_drop)

tweets_selected = tweets_flat[analysis_columns].copy()
tweets_selected['tweet_text_raw'] = tweets_selected['tweet_text'].astype('string')
tweets_selected['clean_text'] = tweets_selected['tweet_text_raw'].map(clean_text)
tweets_selected['text_length'] = tweets_selected['clean_text'].str.len()
tweets_selected['detected_lang'] = tweets_selected['clean_text'].fillna('').map(detect_language)
tweets_selected['is_english'] = tweets_selected['tweet_lang'].eq('en') | tweets_selected['detected_lang'].eq('en')
tweets_selected['lang_match'] = tweets_selected['tweet_lang'].fillna('unknown').eq(tweets_selected['detected_lang'].fillna('unknown'))

tweets_selected.head(5)

,column,dtype,missing_ratio
31,tweet_article,object,1.000000
63,tweet_author_automatedBy,object,1.000000
40,tweet_author_verifiedType,object,1.000000
26,tweet_card,float64,1.000000
30,tweet_communityInfo,object,1.000000
21,tweet_inReplyToId,object,1.000000
27,tweet_quoted_tweet,float64,1.000000
137,tweet_quoted_tweet_article,float64,1.000000
130,tweet_quoted_tweet_author_automatedBy,float64,1.000000
107,tweet_quoted_tweet_author_verifiedType,float64,1.000000


Columnas que se conservan: ['snapshot_bronze_file', 'snapshot__id', 'tweet_id', 'tweet_text', 'tweet_createdAt', 'tweet_lang', 'tweet_source', 'tweet_retweetCount', 'tweet_replyCount', 'tweet_likeCount', 'tweet_quoteCount', 'tweet_viewCount', 'tweet_isReply', 'tweet_conversationId', 'tweet_author_userName', 'tweet_author_name', 'tweet_author_id', 'tweet_author_followers', 'tweet_author_following', 'tweet_author_isVerified', 'tweet_author_isBlueVerified', 'tweet_author_location', 'tweet_author_createdAt']
Columnas que se descartan: ['snapshot_date', 'snapshot_createdAt', 'snapshot_updatedAt', 'snapshot___v', 'tweet_type', 'tweet_url', 'tweet_twitterUrl', 'tweet_bookmarkCount', 'tweet_inReplyToId', 'tweet_displayTextRange', 'tweet_inReplyToUserId', 'tweet_inReplyToUsername', 'tweet_card', 'tweet_quoted_tweet', 'tweet_retweeted_tweet', 'tweet_isLimitedReply', 'tweet_communityInfo', 'tweet_article', 'tweet_author_type', 'tweet_author_url', 'tweet_author_twitterUrl', 'tweet_author_verifiedT

,snapshot_bronze_file,snapshot__id,tweet_id,tweet_text,tweet_createdAt,tweet_lang,tweet_source,tweet_retweetCount,tweet_replyCount,tweet_likeCount,...,tweet_author_isVerified,tweet_author_isBlueVerified,tweet_author_location,tweet_author_createdAt,tweet_text_raw,clean_text,text_length,detected_lang,is_english,lang_match
0,twitter_tweets_20260529_034202.json,6a1851291ca1ed31ad6cd431,2060004476669374743,@Bose 1/ Really disappointed with Bose ending ...,Thu May 28 14:25:14 +0000 2026,en,Twitter for iPhone,0,1,0,...,False,False,London,Mon Nov 29 20:21:06 +0000 2010,@Bose 1/ Really disappointed with Bose ending ...,1/ Really disappointed with Bose ending cloud ...,266,en,True,True
1,twitter_tweets_20260529_034202.json,6a1851291ca1ed31ad6cd431,2059950505447584179,I got into a screaming match on the bus this m...,Thu May 28 10:50:47 +0000 2026,en,Twitter for iPhone,0,0,1,...,False,True,Lagos,Sat Oct 29 10:09:28 +0000 2016,I got into a screaming match on the bus this m...,I got into a screaming match on the bus this m...,1064,en,True,True
2,twitter_tweets_20260529_034202.json,6a1851291ca1ed31ad6cd431,2059950392176234910,I got into a screaming match on the bus this m...,Thu May 28 10:50:20 +0000 2026,en,Twitter for iPhone,0,0,1,...,False,True,Lagos,Sat Oct 29 10:09:28 +0000 2016,I got into a screaming match on the bus this m...,I got into a screaming match on the bus this m...,1064,en,True,True
3,twitter_tweets_20260529_034202.json,6a1851291ca1ed31ad6cd431,2059937735977398654,Ever forget you’re wearing earbuds until the b...,Thu May 28 10:00:02 +0000 2026,en,Twitter for iPhone,0,0,0,...,False,False,"Twin Lake, Michigan",Thu Dec 11 15:54:49 +0000 2008,Ever forget you’re wearing earbuds until the b...,Ever forget you’re wearing earbuds until the b...,142,en,True,True
4,twitter_tweets_20260529_034202.json,6a1851291ca1ed31ad6cd431,2059914754030776460,Dashy talks about Mercules' camera falling on ...,Thu May 28 08:28:43 +0000 2026,en,Twitter for iPhone,4,0,391,...,False,True,Call of Duty League,Sat Dec 09 01:56:00 +0000 2023,Dashy talks about Mercules' camera falling on ...,Dashy talks about Mercules' camera falling on ...,275,en,True,True


### Lectura ejecutiva de la preparacion de tweets

- **Hallazgo principal:** el texto, la fecha, el autor y las señales de interaccion son la base real para EDA y sentimiento.
- **Dato relevante:** `tweet_text_raw` y `clean_text` cumplen funciones distintas, pero el texto canonico debe quedar bien definido antes de Gold.
- **Decision:** la metadata de extraccion y las columnas demasiado tecnicas se quedan como soporte, no como parte central del modelo.